<div style="background: linear-gradient(135deg, #1a237e 0%, #283593 50%, #1565c0 100%); padding: 40px; border-radius: 16px; text-align: center; color: white; margin-bottom: 30px;">
  <h1 style="font-size: 2.2em; font-weight: 900; margin: 0 0 12px 0; letter-spacing: 1px;">🔬 Memory-Less Self-Testing FIR Filter</h1>
  <h2 style="font-size: 1.4em; font-weight: 400; margin: 0 0 20px 0; opacity: 0.9;">Using Vedic Mathematics &amp; Distributed VTU-Based On-Chip Verification</h2>
  <hr style="border-color: rgba(255,255,255,0.3); margin: 20px 0;"/>
  <p style="font-size: 1.1em; margin: 6px 0;">🏫 <strong>SASTRA Deemed University</strong>, Thanjavur, Tamil Nadu, India</p>
  <p style="font-size: 0.95em; opacity: 0.85; margin-top: 14px;">IEEE SSCS Code-a-Chip Travel Grant Award — Submission 2026</p>
</div>

## 👥 Team & Mentors

| Role | Name | Email | Program |
|------|------|-------|---------|
| Student | **Balachandran Harini** | 127004035@sastra.ac.in | 3rd Year B.Tech |
| Student | **Rajeswari N** | 127004202@sastra.ac.in | 3rd Year B.Tech(IEEE Student Member) |
| Student | **Deepthi D** | 127180016@sastra.ac.in | 3rd Year B.Tech |
| Mentor | **Dr. Prabakar T N** | prabakarece@sastra.edu | Associate Professor, ECE |
| Mentor | **Dr. Sriram A** | sriramece@sastra.edu | Associate Professor, ECE |

---
##  Abstract

Traditional Built-In Self-Test (BIST) verification methods for digital arithmetic circuits rely on centralized test hardware or large reference memories, resulting in significant area overhead and limited fault localization capability. This work proposes and implements a **memoryless, distributed on-chip verification framework** using **Verification Test Units (VTUs)** anchored in the principles of **Vedic Mathematics** — specifically the *Anurupyena* sutra for digit-sum casting.

The framework is demonstrated on a **3-tap FIR filter** that integrates:
- A **4×4 Vedic Multiplier** (Urdhva-Tiryagbhyam sutra)
- A **Han–Carlson Parallel Prefix Adder** (8-bit and 9-bit variants)
- Five **VTU blocks** that independently verify each arithmetic operation in real time without any stored reference

The complete RTL-to-GDSII flow was executed using **open-source EDA tools** (Icarus Verilog, Yosys, OpenROAD, Magic VLSI, Netgen) under the **SKY130 PDK**, demonstrating full physical realizability of the design.

---
## 1. Problem Statement & Motivation

### 1.1 The Challenge with Conventional BIST

Conventional Built-In Self-Test (BIST) architectures suffer from three fundamental limitations:

| Limitation | Impact |
|-----------|--------|
| Centralized reference memory | High area overhead (often 20–40% of core area) |
| Coarse-grain fault detection | Cannot localize faults to individual arithmetic units |
| Offline testing model | Cannot verify correctness during live operation |

### 1.2 Proposed Solution

This project introduces a **memoryless, fine-grain verification architecture** where each arithmetic operation — multiplication and addition — is independently verified by a lightweight VTU that uses **digit-sum residue arithmetic** derived from Vedic Mathematics. No stored golden reference is needed; the verification property holds mathematically for every input.

### 1.3 Why Vedic Mathematics?

Vedic Mathematics provides a rich set of algebraic identities that are naturally suited to hardware:

- **Urdhva-Tiryagbhyam** ("vertically and crosswise"): Generates partial products in a structured tree, reducing critical path depth compared to array multipliers.
- **Anurupyena** ("proportionately"): The digit-sum (casting out nines) property — `digit_sum(a × b) = digit_sum(digit_sum(a) × digit_sum(b))` — provides a zero-memory verification oracle for multiplication.
- The same residue property extends to addition: 
`digit_sum(a + b) ≡ digit_sum(digit_sum(a) + digit_sum(b))`.

-**Gunita Samuccaya**is a Vedic mathematics formula (Sutra) used as a, fast, efficient verification technique to test the accuracy of arithmetic and algebraic operations, including multiplication, division, and factorization.
It is based on the principle that the product of the sum of the coefficients of the factors is equal to the sum of the coefficients of the resulting product. 

These properties are **algebraically guaranteed**, making the VTU a provably correct, not heuristic, checker.

---
## 2. System Architecture

### 2.1 Top-Level Block Description

The system implements a 3-tap non-recursive FIR filter:

$$y[n] = a \cdot x[n] + b \cdot x[n-1] + c \cdot x[n-2]$$

where `a`, `b`, `c` are 4-bit coefficients and `x` is a 4-bit input sample.

**Verification layer (VTUs — run in parallel with datapath):**

| VTU | Verifies | Method |
|-----|----------|--------|
| VTU1 | `p0 = x[n] × a` | Digit-sum multiplication residue |
| VTU2 | `p1 = x[n-1] × b` | Digit-sum multiplication residue |
| VTU3 | `p2 = x[n-2] × c` | Digit-sum multiplication residue |
| VTU4 | `s1 = p0 + p1` (8-bit HCA8) | Digit-sum addition residue |
| VTU5 | `y = s1 + p2` (9-bit HCA9) | Digit-sum addition residue |

All five VTU outputs (`v1`–`v5`) are `1` on every valid computation and toggle to `0` upon any arithmetic fault — **no memory required**.

---
## 3. RTL Design — Verilog Implementation

The complete design consists of **9 Verilog modules**. Below we walk through each with commentary.

### 3.1 D Flip-Flop (`d_ff`)
A simple 4-bit synchronous D flip-flop with asynchronous active-high reset. Two instances create the one-cycle and two-cycle delays needed for the FIR taps.

### 3.2 Vedic Multiplier (`vedic_mul`) — Urdhva-Tiryagbhyam

The 4×4 Vedic multiplier implements the **Urdhva-Tiryagbhyam** (UT) sutra. Unlike an array multiplier, UT generates all partial products simultaneously and sums them column-by-column, resulting in a **balanced tree structure** with fewer logic levels.

**Partial product matrix for 4×4:**

Each column is summed using ripple carry, and the carry propagates to the next column — entirely combinational, no clock required.

In [ ]:
vedic_code = """
module vedic_mul(
    input  [3:0] a, b,
    output [7:0] p
);
    // 16 partial products (AND gates)
    wire p00,p01,p02,p03, p10,p11,p12,p13,
         p20,p21,p22,p23, p30,p31,p32,p33;

    assign {p00,p01,p02,p03} = {a[0]&b[0], a[0]&b[1], a[0]&b[2], a[0]&b[3]};
    assign {p10,p11,p12,p13} = {a[1]&b[0], a[1]&b[1], a[1]&b[2], a[1]&b[3]};
    assign {p20,p21,p22,p23} = {a[2]&b[0], a[2]&b[1], a[2]&b[2], a[2]&b[3]};
    assign {p30,p31,p32,p33} = {a[3]&b[0], a[3]&b[1], a[3]&b[2], a[3]&b[3]};

    // Column sums (Urdhva-Tiryagbhyam column accumulation)
    wire [3:0] s1,s2,s3,s4,s5,s6;
    wire       c1,c2,c3,c4,c5,c6;

    assign p[0] = p00;
    assign s1   = p01+p10;          assign p[1]=s1[0]; assign c1=s1[1];
    assign s2   = p02+p11+p20+c1;   assign p[2]=s2[0]; assign c2=s2[1];
    assign s3   = p03+p12+p21+p30+c2; assign p[3]=s3[0]; assign c3=s3[1];
    assign s4   = p13+p22+p31+c3;   assign p[4]=s4[0]; assign c4=s4[1];
    assign s5   = p23+p32+c4;       assign p[5]=s5[0]; assign c5=s5[1];
    assign s6   = p33+c5;           assign p[6]=s6[0]; assign c6=s6[1];
    assign p[7] = c6;
endmodule
"""
print(vedic_code)

### 3.3 Han–Carlson Parallel Prefix Adder (`hca8`, `hca9`)

The **Han–Carlson adder** is a parallel prefix network that computes all carry bits in **O(log₂ N)** levels. It uses generate (`G`) and propagate (`P`) signals:

$$G_i = a_i \cdot b_i \qquad P_i = a_i \oplus b_i$$

The prefix tree merges these over exponentially growing distances across 3 levels (for 8-bit) or 4 levels (for 9-bit), so every carry-in is resolved simultaneously.

**Advantages over ripple-carry adder:**
- 8-bit: ripple-carry requires 8 gate delays; Han–Carlson requires only 3–4
- 9-bit: similarly scales to 4 levels
- Regular wiring structure → friendly to place-and-route

Two variants are instantiated: `hca8` (8-bit in → 9-bit out) and `hca9` (9-bit in → 10-bit out).

### 3.4 Digit-Sum Block (`digit_sum_bc`) — The Vedic Oracle

This is the mathematical heart of the verification system.

**Casting Out Nines Theorem:**
> For any integer `n`, the digit sum `S(n)` (sum of all decimal digits, recursively reduced to a single digit) satisfies:  
> `S(a × b) = S(S(a) × S(b))`  
> `S(a + b) = S(S(a) + S(b))`

This holds modulo 9. The module converts the binary input to BCD, sums digits, and reduces — all using pure combinational logic with no lookup tables or RAM.

**Key insight:** The module handles inputs 0–60 (maximum possible for 4-bit digit-sum products) using a simple if-else BCD decoder — extremely area-efficient.

### 3.5 Top-Level FIR Filter (`fir_filter`)

The top module wires the complete datapath and verification layer. Note that VTU instances run **in parallel** with the datapath — they consume extra area but add zero latency to the critical path.

In [ ]:
print("Module Port Summary — fir_filter")
print("="*50)
ports = [
    ("clk",     "input",  "1",  "System clock"),
    ("reset",   "input",  "1",  "Active-high async reset"),
    ("x",       "input",  "4",  "Current input sample x[n]"),
    ("a",       "input",  "4",  "FIR coefficient a (tap 0)"),
    ("b",       "input",  "4",  "FIR coefficient b (tap 1)"),
    ("c",       "input",  "4",  "FIR coefficient c (tap 2)"),
    ("y",       "output", "10", "Filtered output y[n] (registered)"),
    ("v1",      "output", "1",  "VTU1 valid: x[n]×a correct"),
    ("v2",      "output", "1",  "VTU2 valid: x[n-1]×b correct"),
    ("v3",      "output", "1",  "VTU3 valid: x[n-2]×c correct"),
    ("v4",      "output", "1",  "VTU4 valid: p0+p1 (HCA8) correct"),
    ("v5",      "output", "1",  "VTU5 valid: s1+p2 (HCA9) correct"),
]
print(f"{'Port':<10} {'Dir':<8} {'Width':<7} {'Description'}")
print("-"*60)
for name, direction, width, desc in ports:
    print(f"{name:<10} {direction:<8} {width+'b':<7} {desc}")

---
## 3.5 Top-Level FIR Filter — Full Verilog Code

The cell below shows the **complete RTL** for the `fir_filter` top module (all 9 sub-modules included).
Judges can run this notebook end-to-end: the code cell prints the Verilog, writes it to disk, and then the simulation cell compiles and runs it with **Icarus Verilog**.

In [ ]:
# ── 1. SHOW THE CODE (for readability) ──────────────────────────────────────
fir_verilog = """
// =============================================================================
// Memory-Less Self-Testing FIR Filter
// Vedic Mathematics & Distributed VTU-Based On-Chip Verification
// SASTRA Deemed University — IEEE SSCS Code-a-Chip 2025
// =============================================================================

// ---------------------------------------------------------------------------
// Module 1: D Flip-Flop  (4-bit, async active-high reset)
// ---------------------------------------------------------------------------
module d_ff(
    input clk,
    input reset,
    input [3:0]d,
    output reg [3:0]q
);
    always @(posedge clk or posedge reset) begin
        if (reset)
            q<=4'd0;   
        else
            q<=d;
    end
endmodule
// ---------------------------------------------------------------------------
// Module 2: 4x4 Vedic Multiplier  (Urdhva-Tiryagbhyam)
// ---------------------------------------------------------------------------
module vedic_mul(
    input  [3:0] a,
    input  [3:0] b,
    output [7:0] p
);
    // Partial products
    wire p00,p01,p02,p03;
    wire p10,p11,p12,p13;
    wire p20,p21,p22,p23;
    wire p30,p31,p32,p33;

    // Generate partial products
    assign p00 = a[0]&b[0];
    assign p01 = a[0]&b[1];
    assign p02 = a[0]&b[2];
    assign p03 = a[0]&b[3];
    
    assign p10 = a[1]&b[0];
    assign p11 = a[1]&b[1];
    assign p12 = a[1]&b[2];
    assign p13 = a[1]&b[3];
    
    assign p20 = a[2]&b[0];
    assign p21 = a[2]&b[1];
    assign p22 = a[2]&b[2];
    assign p23 = a[2]&b[3];
    
    assign p30 = a[3]&b[0];
    assign p31 = a[3]&b[1];
    assign p32 = a[3]&b[2];
    assign p33 = a[3]&b[3];

    
    wire [3:0] s1,s2,s3,s4,s5,s6;   // sums (max width)
    wire [3:0] c1,c2,c3,c4,c5,c6;   // carries ( max 2 bit carry is req)

    // Column 0 (no carry)
    assign p[0]=p00;

    // Column 1:
    assign s1 =p01+p10;
	 assign p[1]=s1[0];
    assign c1= s1[1];  

    // Column 2

    assign s2 = p02+p11+p20+c1;
    assign p[2]=s2[0];
    assign c2= s2[1];

    // Column 3
    
    assign s3 = p03+p12+p21+p30+c2;
    assign p[3]=s3[0];
    assign c3=s3[1];  

    // Column 4
    assign s4 =p13 +p22+p31+c3;
    assign p[4]=s4[0];
    assign c4=s4[1];

    // Column 5
    
    assign s5 =p23+p32+c4;
    assign p[5]=s5[0];
    assign c5=s5[1];

     //Column 6
    assign s6 = p33+c5;
    assign p[6]=s6[0];
    assign c6=s6[1];

    // Column 7
    assign p[7]=c6;

endmodule

// ---------------------------------------------------------------------------
// Module 3: Han-Carlson Adder  8-bit in → 9-bit out
// ---------------------------------------------------------------------------
module hca8 (
    input wire [7:0] a,
    input wire [7:0] b,
    output wire [8:0] sum
);
    wire [7:0]g,p;
    wire [7:0]G1,P1;
    wire [7:0]G2,P2;
    wire [7:0]G3,P3;
    wire [8:0]c;

    assign g=a&b;
    assign p=a^b;

    //LEVEL 1
    assign G1[0]=g[0];
    assign P1[0]=p[0];

    genvar i1;
    generate
        for (i1 =1;i1<8;i1=i1+1) begin : L1
            assign G1[i1]=g[i1] |(p[i1]&g[i1-1]);
            assign P1[i1]=p[i1]&p[i1-1];
        end
    endgenerate

    // LEVEL 2
    assign G2[1:0]=G1[1:0];
    assign P2[1:0]=P1[1:0];

    genvar i2;
    generate
        for (i2 =2;i2<8;i2=i2+1) begin : L2
            assign G2[i2]=G1[i2]|(P1[i2]&G1[i2-2]);
        assign P2[i2] =P1[i2]&P1[i2-2];
        end
    endgenerate

    // LEVEL 3
    assign G3[3:0]=G2[3:0];
    assign P3[3:0]=P2[3:0];

    genvar i3;
    generate
        for (i3 =4;i3< 8;i3=i3 + 1) begin : L3
            assign G3[i3]=G2[i3] |(P2[i3]&G2[i3-4]);
            assign P3[i3]=P2[i3]&P2[i3-4];
        end
    endgenerate

    // FINAL 
    wire [7:0] Gf;
    assign Gf=G3;

    //  CARRIES 
    assign c[0]=1'b0;

    genvar i4;
    generate
        for (i4 =0;i4<8;i4=i4 + 1) begin : CARRY
            assign c[i4+1]=Gf[i4];
        end
    endgenerate

    // SUM 
    assign sum ={c[8],(p^c[7:0])};

endmodule

// ---------------------------------------------------------------------------
// Module 4: Han-Carlson Adder  9-bit in → 10-bit out
// ---------------------------------------------------------------------------
module hca9 (
    input  wire[8:0]a,
    input  wire[8:0]b,
    output wire[9:0]sum
);
    // propagate and generate
    wire [8:0]g =a&b;
    wire [8:0]p =a^b;

    // LEVEL 1 (distance 1)
    wire [8:0]G1,P1;
    assign G1[0]=g[0];
    assign P1[0]=p[0];
    genvar i1;
    generate
        for (i1 = 1; i1 < 9; i1 = i1 + 1) begin : L1
            assign G1[i1] =g[i1]|(p[i1]&g[i1-1]);
            assign P1[i1] = p[i1]&p[i1-1];
        end
    endgenerate

    // LEVEL 2 (distance 2)
    wire [8:0]G2,P2;
    assign G2[1:0]=G1[1:0];
    assign P2[1:0]=P1[1:0];
    genvar i2;
    generate
        for (i2 =2;i2<9;i2=i2+1) begin :L2
            assign G2[i2] =G1[i2]|(P1[i2]&G1[i2-2]);
            assign P2[i2]=P1[i2]&P1[i2-2];
        end
    endgenerate

    // LEVEL 3 (distance 4)
    wire [8:0] G3, P3;
    assign G3[3:0]=G2[3:0];
    assign P3[3:0]=P2[3:0];
    genvar i3;
    generate
        for (i3=4;i3<9;i3=i3+1) begin : L3
            assign G3[i3] =G2[i3]|(P2[i3]&G2[i3-4]);
            assign P3[i3] =P2[i3] &P2[i3-4];
        end
    endgenerate

    // LEVEL 4 (distance 8) 
    wire [8:0] G4, P4;

    assign G4[7:0]=G3[7:0];
    assign P4[7:0]=P3[7:0];
    
    assign G4[8]=G3[8]|(P3[8]&G3[0]);
    assign P4[8]=P3[8]&P3[0];

    
    wire [9:0]c;
    assign c[0]=1'b0;
    genvar ic;
    generate
        for (ic=0;ic<9;ic=ic+1) begin : CARRY
            assign c[ic+1] = G4[ic];
        end
    endgenerate

    
    assign sum ={c[9], (p ^ c[8:0])};

endmodule
// ---------------------------------------------------------------------------
// Module 5: Digit-Sum Block  (Anurupyena / casting-out-nines)
// Converts binary input to single decimal digit-sum (mod 9)
// Handles inputs 0-60 (max possible for 4-bit products)
// ---------------------------------------------------------------------------
module digit_sum_bc                                        
 #(                                                        
    parameter WIDTH = 16                                   
)(                                                         
    input  [WIDTH-1:0] in,   // only values 0?60 are expected
    output reg [3:0] ds                                    
);                                                         
                                                           
    reg [3:0] tens, ones;                                  
                                                           
    always @(*) begin                                      
        tens = 4'b0000;                                    
        ones = 4'b0000;                                    
        ds   = 4'b0000;                                    
                                                           
        // Fast path: 0?8                                  
                                                           
                                                           
            // BCD separation (0?60 only)                  
            if (in < 16'd10) begin            // 9         
                tens = 4'b0000;                            
                ones = in[3:0];                            
            end                                            
            else if (in < 16'd20) begin       // 10?19     
                tens = 4'b0001;                            
                ones = in - 16'd10;                        
            end                                            
            else if (in < 16'd30) begin       // 20?29     
                tens = 4'b0010;                            
                ones = in - 16'd20;                        
            end                                            
            else if (in < 16'd40) begin       // 30?39     
                tens = 4'b0011;                            
                ones = in - 16'd30;                        
            end                                            
            else if (in < 16'd50) begin       // 40?49     
                tens = 4'b0100;                            
                ones = in - 16'd40;                        
                                                           
                                                           
            end                                            
            else if (in < 16'd60) begin       // 40?49     
                tens = 4'b0101;                            
                ones = in - 16'd50;                        
            end                                            
            else begin                        // 50?60     
                tens = 4'b0110;                            
                ones = in - 16'd60;                        
            end                                            
                                                           
                                                           
            // Digit sum (max = 5 + 9 = 14)                
            ds = tens + ones;                              
                                                           
             // Final reduction to single digit            
    if (ds >= 10)                                          
        ds = ds - 9;                                       
        end                                                
                                                           
                                                           
endmodule                  

// ---------------------------------------------------------------------------
// Module 6: VTU for Multiplication  — checks DS(p) == DS(DS(a)*DS(b))
// ---------------------------------------------------------------------------
module mul_verification(
    input [3:0]a,
    input [3:0]b,
    input [7:0]product,
    output valid
);
    wire [3:0]ds_a, ds_b, ds_prod;
    wire [7:0]prod_ds_mul;
    wire [3:0]ds_expected;

    
    digit_sum_bc #(4)  DS_A  (a,ds_a);
    digit_sum_bc #(4)  DS_B  (b,ds_b);
    digit_sum_bc #(8)  DS_P  (product,ds_prod);

    // multiply 
    assign prod_ds_mul=ds_a *ds_b;  

    
    digit_sum_bc #(8) DS_EXP (prod_ds_mul,ds_expected);

    assign valid =(ds_prod==ds_expected);

endmodule

// ---------------------------------------------------------------------------
// Module 7: VTU for 8-bit Addition  — checks DS(s) == DS(DS(a)+DS(b))
// ---------------------------------------------------------------------------
module adder_verification(
    input  [7:0]a,
    input  [7:0]b,
    input  [8:0]sum,   //hca  9 bit o/p
    output valid
);
    wire [3:0]ds_a,ds_b,ds_sum;
    wire [5:0]ds_add;   // max width
    wire [3:0]ds_expected;

    digit_sum_bc #(8) DS_A(a,ds_a);
    digit_sum_bc #(8) DS_B(b,ds_b);
    digit_sum_bc #(9) DS_S(sum,ds_sum);

    assign ds_add =ds_a+ds_b;     

    digit_sum_bc #(8) DS_EXP ({2'b00, ds_add},ds_expected);


    assign valid = (ds_sum==ds_expected);

endmodule

// ---------------------------------------------------------------------------
// Module 8: VTU for 9-bit Addition  — checks DS(s) == DS(DS(a)+DS(b))
// ---------------------------------------------------------------------------
module adder_verification_9bit(
    input  [8:0] a,      
    input  [8:0] b,      
    input  [9:0] sum,    
    output valid
);
    wire [3:0] ds_a,ds_b,ds_sum;
    wire [6:0] ds_add;   
    wire [3:0] ds_expected;

    digit_sum_bc #(9) DS_A(a,ds_a);
    digit_sum_bc #(9) DS_B(b,ds_b);
    digit_sum_bc #(10) DS_S(sum,ds_sum);

    assign ds_add=ds_a+ds_b;   // 0..18

    digit_sum_bc #(8) DS_EXP ({1'b0, ds_add},ds_expected);


    assign valid =(ds_sum==ds_expected);

endmodule


// ---------------------------------------------------------------------------
// Module 9: TOP — fir_filter
// y[n] = a*x[n] + b*x[n-1] + c*x[n-2]
// ---------------------------------------------------------------------------

module fir_filter(
    input clk,
    input reset,
    input [3:0] x,
    input [3:0] a,
    input [3:0] b,
    input [3:0] c,
    output reg [9:0] y,
    output v1,v2,v3,v4,v5
	 //output verif_pass
);
    
    // Delays x1=x(n-1)  x2=x(n-2)
    wire [3:0] x1, x2;
	 
    
    // delay using dff
    d_ff d1(clk, reset, x, x1);
    d_ff d2(clk, reset, x1, x2);
    
    // vedic multiplier
    wire [7:0] p0, p1, p2;
    
    vedic_mul m1(x, a, p0);
    vedic_mul m2(x1, b, p1);
    vedic_mul m3(x2, c, p2);
    
    // han-carlson adder
    wire [8:0] s1;
    hca8 h1(p0, p1, s1);
    wire [9:0] y_comb;
    hca9 h2(s1, {1'b0, p2}, y_comb);
    
    
    always @(posedge clk or posedge reset) begin
        if (reset)
            y <= 0;
        else
            y <= y_comb;
    end
    
    // VTU
    //wire v1, v2, v3, v4, v5;
    mul_verification vtu1( x, a, p0, v1);
    mul_verification vtu2( x1, b, p1, v2);
    mul_verification vtu3( x2, c, p2, v3);
    
    
    adder_verification vtu4( p0, p1, s1, v4);
    adder_verification_9bit vtu5(s1,{1'b0, p2},y_comb, v5);
    
    
    //assign verif_pass = v1 & v2 & v3 & v4 & v5;

endmodule

"""
print(fir_verilog)


In [ ]:
# ── 2. WRITE CODE TO FILE  ──────────────────────────────────────────────────
with open("fir_filter.v", "w") as f:
    f.write(fir_verilog)
print("fir_filter.v written successfully.")

# ── Testbench ────────────────────────────────────────────────────────────────
tb_code = """

`timescale 1ns/1ps

module tb_fir_fil;
    reg clk;
    reg reset;
    reg [3:0] x;
    reg [3:0] a, b, c;

    wire [9:0] y;
    wire v1, v2, v3, v4, v5;

    fir_filter DUT (
        .clk   (clk),
        .reset (reset),
        .x     (x),
        .a     (a),
        .b     (b),
        .c     (c),
        .y     (y),
        .v1    (v1),
        .v2    (v2),
        .v3    (v3),
        .v4    (v4),
        .v5    (v5)
    );
    always #5 clk = ~clk;
    task apply_x;
        input [3:0] xv;
        integer i;
        begin
            @(negedge clk);
            x = xv;

            // wait enough cycles for 3-tap FIR to settle
            for (i = 0; i < 4; i = i + 1) begin
                @(posedge clk);
                $display(
                    "T=%0t | x=%0d | y=%0d | VTU=%b%b%b%b%b",
                    $time, x, y, v1, v2, v3, v4, v5
                );
            end
            $display("----------------------------------");
        end
    endtask

    // --------------------------------------------------
    // Test sequence
    // --------------------------------------------------
    initial begin
        // Init
        clk   = 0;
        reset = 1;
        x     = 0;

        // FIR coefficients
        a = 4'd2;
        b = 4'd1;
        c = 4'd3;

        // Hold reset for 2 clock cycles
        repeat (2) @(posedge clk);
        reset = 0;

        $display("\n=== FIR FILTER TEST START ===");
        $display("Coefficients: a=2, b=1, c=3");
        $display("----------------------------------");

        // -------- Test vectors --------
        apply_x(4'd1);   // expect: 2,3,6
        apply_x(4'd3);   // expect: 6,9,18
        apply_x(4'd2);   // expect: 4,5,11
        apply_x(4'd4);   // expect: 8,10,20
        apply_x(4'd0);   // pipeline decay
        apply_x(4'd5);   // expect: 10,15,30

        $display("=== TEST COMPLETE ===");
        $stop;
    end

endmodule
"""

with open("fir_tb.v", "w") as f:
    f.write(tb_code)
print("fir_tb.v written successfully.")


In [ ]:
# ── 3. RUN SIMULATION with Icarus Verilog ───────────────────────────────────
import subprocess

# Compile
compile_result = subprocess.run(
    ["iverilog", "-o", "fir_tb", "fir_filter.v", "fir_tb.v"],
    capture_output=True, text=True
)

if compile_result.returncode != 0:
    print("COMPILATION ERRORS:")
    print(compile_result.stderr)
else:
    print("Compilation: SUCCESS")
    # Simulate
    sim_result = subprocess.run(
        ["vvp", "fir_tb"],
        capture_output=True, text=True
    )
    print("\nSimulation output:")
    print(sim_result.stdout)
    if sim_result.stderr:
        print("Warnings:", sim_result.stderr)


---
## 4. Simulation & Verification

### 4.1 Simulation Environment

Simulation was performed using **Icarus Verilog (iverilog)** and waveforms were inspected in **GTKWave** and also in **ModelSim** .

**Compile & simulate commands:**

```bash
# Compile
iverilog -o fir_sim fir_filter.v tb_fir_filter.v

# Simulate (generates .vcd)
vvp fir_sim

# View waveforms
gtkwave dump.vcd &
```



---
## 5. RTL-to-GDSII Physical Design Flow

The complete physical implementation was performed using **100% open-source EDA tools** under the **SkyWater SKY130 130nm PDK**.

### 5.1 Tool Flow Summary

| Stage | Tool | Purpose | Status |
|-------|------|---------|--------|
| 1–2. RTL & TB | Icarus Verilog | Compile & simulate | ✅ Done |
| 3. IO Config | OpenLane/ORFS | Pad configuration | ✅ Done |
| 4. Simulation | GTKWave | Waveform viewing (.vcd) | ✅ Done |
| 5. Synthesis | Yosys | RTL → Gate-level netlist | ✅ Done |
| 6. Floorplan | OpenROAD | Die/core area, pin placement | ✅ Done |
| 7. Placement | OpenROAD | Standard cell placement | ✅ Done |
| 8. CTS | OpenROAD (TritonCTS) | Clock tree synthesis | ✅ Done |
| 9. Routing | OpenROAD (TritonRoute) | Metal routing | ✅ Done |
| 10. DRC | Magic VLSI | Design rule checking | ✅ Done |
| 11. LVS | Netgen | Layout vs Schematic | ✅ Done |
| 12. Antenna | OpenLane/Magic | Antenna rule check | ✅ Done |
| 13. GDSII | Magic stream out | Final GDSII export | ✅ Done |

### 5.2 Synthesis (Yosys)

```bash
# Yosys synthesis script
read_verilog fir_filter.v
synth -top fir_filter
dfflibmap -liberty sky130_fd_sc_hd__tt_025C_1v80.lib
abc -liberty sky130_fd_sc_hd__tt_025C_1v80.lib
write_verilog -noattr synth/fir_filter_synth.v
stat
```

### 5.3 OpenROAD Flow
    #automated flow
    cd ~/eda_tools/OpenROAD-flow-scripts source env.sh 
    # Copy your design files to the flow structure:
     mkdir -p flow/designs/sky130hd/counter_4bit cp ~/counter_project/rtl/counter_4bit.v flow/designs/sky130hd/counter_4bit/ cp ~/counter_project/config.json flow/designs/sky130hd/counter_4bit/ 
     # Run the entire flow with one command:
      cd flow make DESIGN_CONFIG=designs/sky130hd/counter_4bit/config.mk
       # Open the final result in OpenROAD GUI: make gui 

### 5.4 Physical Verification (Magic + Netgen)

```bash
# DRC in Magic
magic -rcfile sky130A.magicrc
# In Tcl console:
load fir_filter.mag
drc check
drc why

# LVS with Netgen
netgen -batch lvs "fir_filter.spice fir_filter" \
    "fir_filter_synth.v fir_filter" sky130A_setup.tcl
```

---
## 6. Results

### 6.1 Synthesis Statistics (Yosys + SKY130)

In [ ]:
# Synthesis and timing results summary
=== fir_filter ===

   Number of wires:               1392
   Number of wire bits:           2117
   Number of public wires:         191
   Number of public wire bits:     916
   Number of ports:                 12
   Number of port bits:             33
   Number of memories:               0
   Number of memory bits:            0
   Number of processes:              0
   Number of cells:               1413
 Chip area for module '\fir_filter': 9233.856000
     of which used for sequential elements: 450.432000 (4.88%)



---
## 7. Key Innovations & Contributions

This project makes the following original contributions:

### 7.1 Vedic Mathematics as a Verification Mechanism
Prior work uses Vedic Mathematics exclusively for **computation** (fast multipliers). This project is among the first to exploit Vedic algebraic properties (casting-out-nines) as a **verification oracle** for digital arithmetic circuits.

### 7.2 Memoryless Architecture
The VTU requires **zero stored reference values**. The verification property `DS(a×b) = DS(DS(a)×DS(b))` holds unconditionally for every pair of inputs — the checker logic is purely combinational and self-contained.

### 7.3 Distributed Fault Localization
Each VTU (`v1`–`v5`) independently monitors exactly one arithmetic unit. A fault in `p0` (VTU1 = 0) is immediately distinguished from a fault in `s1` (VTU4 = 0), enabling **fine-grain localization** impossible with a single BIST wrapper.

### 7.4 Heterogeneous Arithmetic Architecture
The combination of a **Vedic multiplier** (Urdhva-Tiryagbhyam) and a **Han–Carlson parallel prefix adder** within the same pipeline is architecturally coherent: both components minimize critical path depth through structural regularity, and both are amenable to the digit-sum residue verification method.

### 7.5 Full Open-Source Silicon Flow
The design was fully implemented from RTL to GDSII using open-source tools (Icarus Verilog, Yosys, OpenROAD, Magic, Netgen) on the SkyWater SKY130 PDK — demonstrating the accessibility and completeness of modern open-source silicon design.

---

### 8 Future Enhancements

- **Dual-residue checking:** Combining mod-9 and mod-11 digit sums reduces aliasing probability to ~1/99
- **VTU for wider datapaths:** Extension to 8×8 or 16×16 multipliers using recursive Vedic decomposition
- **Power analysis:** Formal power estimation of the VTU overhead using OpenROAD's power analysis flow


---
## 9. Conclusion

This work presents a novel **memoryless, distributed on-chip verification architecture** for a 3-tap FIR filter, demonstrating that principles from ancient Vedic Mathematics can be directly translated into lightweight, hardware-efficient error detection logic for modern digital circuits.

The five VTU blocks add modest area overhead while providing **independent, real-time verification** of every arithmetic operation — three multiplications and two additions — without any stored reference values. The complete RTL-to-GDSII implementation on the SkyWater SKY130 130nm PDK using entirely open-source tools demonstrates that the design is physically realizable.

By combining the **Urdhva-Tiryagbhyam Vedic multiplier**, the **Han–Carlson parallel prefix adder**, and **digit-sum residue VTUs**, this project bridges a 2500-year-old mathematical tradition with contemporary silicon design — making a case that mathematical elegance and hardware efficiency are not in conflict.

---
<div style="background:#f0f4f8; padding: 16px; border-left: 4px solid #1565c0; border-radius: 4px; margin-top: 20px;">
<strong>Acknowledgements:</strong> The authors thank the IEEE SSCS for the Code-a-Chip initiative that makes silicon design accessible to students worldwide, the SkyWater Technology and Google for the open-source SKY130 PDK, and the OpenROAD and OpenLane communities for their invaluable open-source EDA infrastructure.
</div>

---
## References

1. Bharati Krishna Tirthaji, *Vedic Mathematics*, Motilal Banarsidass Publishers, 1965.
2. T. Han and C. A. Carlson, "Fast area-efficient VLSI adders," *Proc. 8th Symp. Computer Arithmetic*, 1987, pp. 49–56.
3. REAL-TIME VEDIC MATHEMATICS BASED MEMORYLESS ARITHMETIC CIRCUITS VERIFICATION TECHNIQUE, Dr.THAMMAMPATTI NATARAJAN PRABAKAR
4. M. Nicolaidis, "Carry checking / parity prediction adders and ALUs," *IEEE TVLSI*, vol. 11, no. 1, 2003.
5. A. Kahng et al., "OpenROAD: Toward a Self-Driving, Open-Source Digital Layout Implementation Tool Chain," *Proc. DAC*, 2021.
6. SkyWater Technology, *SKY130 PDK Documentation*, https://skywater-pdk.readthedocs.io, 2020.
7. IEEE SSCS, *Code-a-Chip Travel Grant Award*, https://sscs.ieee.org/education/code-a-chip, 2025.
